### Human in the Loop (HITL) Example with CrewAI
- USE CASE: Intelligent Payment Authorization System
    - Agent reviews payment requests and executes or 
          holds transfers based on natural language approval 
          from a human reviewer — no hardcoded conditions.
       

In [2]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

load_dotenv()

llm = LLM(
    model="gemini/gemini-2.5-pro",   
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

# ── Tools ─────────────────────────────────────────────────

@tool("Transfer Amount")
def transfer_amount(amount: str, receiver: str) -> str:
    """Transfers the specified amount to the receiver.
    Call this when payment should be approved and processed."""
    print(f"\n💸 EXECUTING TRANSFER: ₹{amount} → {receiver}")
    return f"✅ Transfer successful! ₹{amount} sent to {receiver}. Transaction ID: TXN{os.urandom(4).hex().upper()}"


@tool("Hold Transfer")
def hold_transfer(amount: str, receiver: str, reason: str) -> str:
    """Holds and blocks the transfer for the given reason.
    Call this when payment should be stopped or flagged."""
    print(f"\n🚫 HOLDING TRANSFER: ₹{amount} → {receiver}")
    return f"⛔ Transfer of ₹{amount} to {receiver} has been HELD. Reason: {reason}. Ref: HOLD{os.urandom(4).hex().upper()}"


# ── Agent ─────────────────────────────────────────────────

payment_agent = Agent(
    role="Payment Processing Agent",
    goal="""Review payment requests and human feedback carefully.
            Based on human's instruction, decide whether to 
            transfer or hold the payment using available tools.
            Once you receive human feedback, call the tool ONCE
            and STOP. Do not ask for feedback again.""",
    backstory="""You are a smart payment processor. You process payments
                 based on human reviewer's feedback and instructions.
                 You never transfer money without proper human approval.
                 You use your tools to either execute or hold transfers.""",
    tools=[transfer_amount, hold_transfer],
    llm=llm,
    max_iter=3
)

# ── Task ──────────────────────────────────────────────────

payment_task = Task(
    description="""
    A payment request has come in with below details:
    - Amount    : {amount}
    - Sender    : {sender}
    - Receiver  : {receiver}
    - Purpose   : {purpose}

    Wait for human reviewer's feedback.
    Based on the feedback, call the correct tool EXACTLY ONCE.
    After tool returns result, immediately return final answer.
    DO NOT ask for human input again after calling the tool.
    """,
    expected_output="""A clear confirmation of what action was taken — 
                       either transfer executed or transfer held, 
                       with transaction reference.""",
    agent=payment_agent,
    human_input=True
)

crew = Crew(
    agents=[payment_agent],
    tasks=[payment_task]
)

# ── Take inputs dynamically ───────────────────────────────

print("\n" + "="*60)
print("       PAYMENT AUTHORIZATION SYSTEM")
print("="*60)
amount   = input("  Enter Amount   : ₹")
sender   = input("  Enter Sender   : ")
receiver = input("  Enter Receiver : ")
purpose  = input("  Enter Purpose  : ")

PAYMENT_DETAILS = {
    "amount"  : amount,
    "sender"  : sender,
    "receiver": receiver,
    "purpose" : purpose
}

print("="*60)
print("  📋 Payment Request Submitted for Review:")
print(f"  Amount   : ₹{amount}")
print(f"  Sender   : {sender}")
print(f"  Receiver : {receiver}")
print(f"  Purpose  : {purpose}")
print("="*60)
print("  ⚠️  Provide your decision when asked.")
print("  Examples: 'approve it' / 'hold it, looks suspicious'")
print("="*60 + "\n")

result = crew.kickoff(inputs=PAYMENT_DETAILS)

print("\n" + "="*60)
print("  FINAL RESULT")
print("="*60)
print(result)


       PAYMENT AUTHORIZATION SYSTEM
  📋 Payment Request Submitted for Review:
  Amount   : ₹50000000000000
  Sender   : Jim
  Receiver : Srini
  Purpose  : Transfer
  ⚠️  Provide your decision when asked.
  Examples: 'approve it' / 'hold it, looks suspicious'


🚫 HOLDING TRANSFER: ₹50000000000000 → Srini


╭────────────────────────────────────────── 💬 Human Feedback Required ───────────────────────────────────────────╮
│                                                                                                                 │
│  Provide feedback on the Final Result above.                                                                    │
│                                                                                                                 │
│  • If you are happy with the result, simply hit Enter without typing anything.                                  │
│  • Otherwise, provide specific improvement requests.                                                            │
│  • You can provide multiple rounds of feedback until satisfied.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Processing your feedback...


🚫 HOLDING TRANSFER: ₹50000000000000 → Srini


╭────────────────────────────────────────── 💬 Human Feedback Required ───────────────────────────────────────────╮
│                                                                                                                 │
│  Provide feedback on the Final Result above.                                                                    │
│                                                                                                                 │
│  • If you are happy with the result, simply hit Enter without typing anything.                                  │
│  • Otherwise, provide specific improvement requests.                                                            │
│  • You can provide multiple rounds of feedback until satisfied.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


  FINAL RESULT

⛔ Transfer of ₹50000000000000 to Srini has been HELD. Reason: Hold as per human reviewer's feedback.. Ref: HOLDFA659990
